# Topic 33 — Recurrent Neural Networks (RNNs)
### Theory → from-scratch RNN cell → PyTorch nn.RNN → the vanishing gradient problem, visualized.

CNNs (Topic 32) are great for spatial data (images). Text is **sequential** — word order matters,
and a word's meaning often depends on words far earlier in the sentence. An **RNN** processes a
sequence one element at a time, carrying a **hidden state** forward that acts as a compressed
memory of everything seen so far.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Sequence & hidden state — the core idea

At each timestep `t`, the RNN combines the new input `x_t` with its previous hidden state
`h_{t-1}` to produce a new hidden state `h_t`:

```text
h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b)
```

The SAME weights (`W_xh`, `W_hh`) are reused at every timestep — this is the "recurrent connection".
`h_t` is passed both to the next timestep AND (optionally) used to produce an output at time `t`.

In [ ]:
def rnn_cell_step(x_t, h_prev, W_xh, W_hh, b):
    return np.tanh(x_t @ W_xh + h_prev @ W_hh + b)

# A tiny made-up sequence: 4 timesteps, each input is a 3-dim vector
seq_len, input_dim, hidden_dim = 4, 3, 5
rng = np.random.default_rng(0)
X_seq = rng.normal(0, 1, size=(seq_len, input_dim))

W_xh = rng.normal(0, 0.3, size=(input_dim, hidden_dim))
W_hh = rng.normal(0, 0.3, size=(hidden_dim, hidden_dim))
b = np.zeros(hidden_dim)

h = np.zeros(hidden_dim)   # initial hidden state, usually all zeros
hidden_states = [h]
for t in range(seq_len):
    h = rnn_cell_step(X_seq[t], h, W_xh, W_hh, b)
    hidden_states.append(h)
    print(f"timestep {t}: hidden state = {np.round(h, 3)}")

# Notice: h at each step depends on ALL previous inputs, compressed into one fixed-size vector --
# that's the RNN's "memory".

## 2. Recurrent connection, unrolled

"Unrolling" an RNN means drawing/thinking of it as one cell repeated across time, each copy
sharing the SAME weights but receiving a different input and the previous copy's hidden state:

```text
x1 -> [RNN cell] -> h1 -> [RNN cell] -> h2 -> [RNN cell] -> h3 -> [RNN cell] -> h4
       ^ same weights every step ^

In [ ]:
plt.figure(figsize=(9, 2.5))
for t in range(seq_len):
    plt.scatter(t, 0, s=800, color="lightblue", edgecolor="black", zorder=2)
    plt.text(t, 0, f"cell\n(t={t})", ha="center", va="center", fontsize=8)
    if t > 0:
        plt.annotate("", xy=(t-0.15, 0), xytext=(t-0.85, 0),
                     arrowprops=dict(arrowstyle="->", color="red"))
    plt.annotate("", xy=(t, -0.5), xytext=(t, -0.05),
                 arrowprops=dict(arrowstyle="<-", color="gray"))
    plt.text(t, -0.6, f"x{t}", ha="center")
plt.xlim(-0.7, seq_len - 0.3); plt.ylim(-1, 0.5)
plt.axis("off")
plt.title("Unrolled RNN — red arrows = hidden state passed between timesteps (SAME weights reused)")
plt.show()

## 3. PyTorch's `nn.RNN`

In [ ]:
rnn_layer = nn.RNN(input_size=3, hidden_size=5, batch_first=True)

X_batch = torch.tensor(X_seq, dtype=torch.float32).unsqueeze(0)   # add batch dim -> (1, seq_len, input_dim)
print("input shape:", X_batch.shape, "-> (batch, seq_len, input_dim)")

output, final_hidden = rnn_layer(X_batch)
print("output shape:", output.shape, "-> hidden state at EVERY timestep")
print("final_hidden shape:", final_hidden.shape, "-> just the LAST hidden state")
print("\nlast timestep's output equals final_hidden:", torch.allclose(output[0, -1], final_hidden[0, 0]))

## 4. A simple sequence classification example

Classifying whether a short sequence of numbers is trending "up" or "down" overall — a toy
stand-in for "classify a sequence of word embeddings as bullying/not" (which you'll build in
Topic 34 with LSTM, the RNN's stronger successor).

In [ ]:
def make_trend_sequence(length=6, trending_up=True):
    base = rng.normal(0, 0.3, size=length)
    trend = np.linspace(0, 2, length) if trending_up else np.linspace(2, 0, length)
    return (base + trend).reshape(-1, 1).astype(np.float32)

X_data = np.array([make_trend_sequence(trending_up=bool(i % 2)) for i in range(100)])
y_data = np.array([i % 2 for i in range(100)])   # 1 = up, 0 = down

X_tensor = torch.tensor(X_data)
y_tensor = torch.tensor(y_data, dtype=torch.long)

class TrendRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=8, n_classes=2):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, h_n = self.rnn(x)             # h_n: final hidden state, shape (1, batch, hidden_size)
        h_n = h_n.squeeze(0)              # -> (batch, hidden_size)
        return self.fc(h_n)               # classify based on the FINAL hidden state

model = TrendRNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

X_tensor, y_tensor = X_tensor.to(device), y_tensor.to(device)
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(5, 4))
plt.plot(losses)
plt.title("RNN training loss (trend classification)")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.show()

preds = model(X_tensor).argmax(1)
acc = (preds == y_tensor).float().mean().item()
print("training accuracy:", acc)

## 5. The vanishing gradient problem — why plain RNNs struggle with LONG sequences

During backpropagation THROUGH TIME, gradients get multiplied by the same weight matrix repeatedly
(once per timestep). If those values are consistently < 1, the gradient shrinks exponentially as it
flows backward through many timesteps — early timesteps barely get any learning signal at all.
This is exactly why LSTM (Topic 34) was invented.

In [ ]:
# Simulate the compounding effect: repeatedly multiplying by a value < 1 (vanishing)
# or > 1 (exploding), the same way gradients compound across RNN timesteps
timesteps = np.arange(1, 51)

vanishing = 0.9 ** timesteps    # gradient shrinking each step (common with tanh/sigmoid activations)
exploding = 1.1 ** timesteps    # gradient growing each step (can happen with poorly-scaled weights)
stable = 1.0 ** timesteps       # what we'd WANT: gradient magnitude staying roughly constant

plt.figure(figsize=(7, 4))
plt.plot(timesteps, vanishing, label="vanishing (factor=0.9 per step)")
plt.plot(timesteps, exploding, label="exploding (factor=1.1 per step)")
plt.plot(timesteps, stable, label="stable (factor=1.0 per step)", linestyle="--")
plt.yscale("log")
plt.xlabel("number of timesteps back-propagated through")
plt.ylabel("relative gradient magnitude (log scale)")
plt.legend()
plt.title("Vanishing / exploding gradients over a long sequence")
plt.show()
# After ~30-40 timesteps, the vanishing gradient is already many orders of magnitude smaller --
# the network effectively can't learn dependencies that span that many steps.
# LSTM's gating mechanism (Topic 34) is specifically designed to fix this.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Increase make_trend_sequence's length to 20 and 50 -- does the RNN's training accuracy drop
#    as sequences get longer (a symptom of vanishing gradients)?
# 2. Change hidden_size in TrendRNN from 8 to 32 -- does accuracy or training speed change?
# 3. In the vanishing/exploding simulation, try factor=0.99 instead of 0.9 -- how many more
#    timesteps does it take before the gradient shrinks to the same tiny magnitude?
# 4. Look at nn.RNN's `nonlinearity` parameter (default 'tanh', can be 'relu') -- read the docs
#    and note in one sentence why tanh's bounded output (-1 to 1) contributes to vanishing gradients.

---
### Next up: **Topic 34 — LSTM** — very important for your NLP path.

Say "next" when you're ready.